# 04 Embedder Forward Pass

Goal: verify the style embedder can consume Stage 1 triplets and produce normalized 512d embeddings. No training happens in this notebook.

If dependencies are missing, run this once. It works even if the remote kernel is not started from the repo root:

```python
import sys, subprocess
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
req = next((p / "requirements.txt" for p in candidates if (p / "requirements.txt").exists()), None)
cmd = [sys.executable, "-m", "pip", "install"]
cmd += ["-r", str(req)] if req else ["torch", "transformers", "openai", "python-dotenv"]
subprocess.check_call(cmd)
```

In [ ]:
import os
from pathlib import Path
import sys

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path in this kernel.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoTokenizer

from voice_clone.data.dataset import TripletTextDataset
from voice_clone.models import StyleEmbedder, count_parameters

TRIPLETS_PATH = ROOT / "data" / "processed" / "triplets_dry_run.jsonl"
MODEL_NAME = "allenai/longformer-base-4096"
BATCH_SIZE = 2
MAX_LENGTH = 512

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("ROOT:", ROOT)
print("TRIPLETS_PATH:", TRIPLETS_PATH)
device

In [ ]:
dataset = TripletTextDataset.from_jsonl(TRIPLETS_PATH, seed=7)
examples = [dataset[i] for i in range(BATCH_SIZE)]

for ex in examples:
    print(ex["author_id"], ex["anchor_id"], "positive:", ex["positive_id"])

len(dataset)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = StyleEmbedder.from_pretrained(MODEL_NAME).to(device)
model.eval()

count_parameters(model)

In [ ]:
anchors = [ex["anchor"] for ex in examples]
positives = [ex["positive"] for ex in examples]
negatives = [ex["negative"] for ex in examples]

encoded = tokenizer(
    anchors + positives + negatives,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
encoded = {k: v.to(device) for k, v in encoded.items()}

print({k: tuple(v.shape) for k, v in encoded.items()})

In [ ]:
with torch.no_grad():
    embeddings = model(**encoded)

anchor_emb, positive_emb, negative_emb = embeddings.chunk(3, dim=0)

print("embedding shape:", tuple(embeddings.shape))
print("embedding norms:", embeddings.norm(dim=-1).detach().cpu().round(decimals=4).tolist())
print("anchor-positive cosine:", torch.sum(anchor_emb * positive_emb, dim=-1).detach().cpu().tolist())
print("anchor-negative cosine:", torch.sum(anchor_emb * negative_emb, dim=-1).detach().cpu().tolist())

assert embeddings.shape == (BATCH_SIZE * 3, 512)
assert torch.isfinite(embeddings).all()
assert torch.allclose(embeddings.norm(dim=-1), torch.ones(BATCH_SIZE * 3, device=device), atol=1e-4)

print("forward pass ok")